# AutoGen with Custom Models (Tencent Cloud/Deepseek)

In this code sample, you will use the [AutoGen](https://aka.ms/ai-agents/autogen) AI Framework with custom models like Tencent Cloud or Deepseek that support OpenAI-compatible APIs. 

The goal of this sample is to show you how to configure AutoGen to work with your own model provider instead of GitHub Models.

## Import the Needed Python Packages 

In [ ]:
# 在您的 ipynb 文件中使用
from model_adapter import CustomModelAdapter, create_default_adapter

# 方法1：使用默认配置（从环境变量读取）
adapter = create_default_adapter()

# 方法2：自定义配置
adapter = CustomModelAdapter(
    api_key="sk-SeshgPrCkDuqkPHCNfLkxaPiV4jasvAN7qKjnXgzTv7Dbos9",
    endpoint="https://api.lkeap.cloud.tencent.com/v1",
    model_id="deepseek-v3-0324"
)

# 测试连接
if adapter.test_connection():
    print("连接成功！")

ModuleNotFoundError: No module named 'model_adapter'

## Create the Client 

In this sample, we will use a custom model API compatible with OpenAI format, such as Tencent Cloud or Deepseek. 

The `model` is defined from your environment variables. You can change the model to another model available on your provider to see the different results. 

As a quick test, we will just run a simple prompt - `What is the capital of France`. 

In [ ]:
load_dotenv()

# Get configuration from custom model adapter
ag_config = get_autogen_config()

# Verify configuration
api_key = os.getenv("OPENAI_API_KEY", os.getenv("GITHUB_TOKEN"))
endpoint = os.getenv("OPENAI_ENDPOINT", os.getenv("GITHUB_ENDPOINT"))
model = os.getenv("OPENAI_CHAT_MODEL_ID", os.getenv("GITHUB_MODEL_ID"))

if not all([api_key, endpoint, model]):
    print("❌ Error: Missing required environment variables!")
    print(f"  API Key: {'Set' if api_key else 'Missing'}")
    print(f"  Endpoint: {'Set' if endpoint else 'Missing'}")
    print(f"  Model: {'Set' if model else 'Missing'}")
else:
    print(f"✅ Using configuration:")
    print(f"  Endpoint: {endpoint}")
    print(f"  Model: {model}")
    print(f"  API Key: {'*' * (len(api_key) - 4) + api_key[-4:] if api_key else 'Not set'}")

# Create client using environment configuration
try:
    client = AzureAIChatCompletionClient(
        model=model,
        endpoint=endpoint,
        credential=AzureKeyCredential(api_key),
        model_info={
            "json_output": True,
            "function_calling": True,
            "vision": False,  # Set to True if your model supports vision
            "family": "openai-compatible",
        },
    )

    result = await client.create([UserMessage(content="What is the capital of France?", source="user")])
    print(f"✅ Test query successful: {result}")
except Exception as e:
    print(f"❌ Error creating client: {e}")

## Defining the Agent 

Now that we have set up the `client` and confirmed that it is working, let us create an `AssistantAgent`. Each agent can be assigned a: 
**name** - A short hand name that will be useful in referencing it in multi-agent flows. 
**model_client** - The client that you created in the earlier step. 
**tools** - Available tools that the Agent can use to complete a task.
**system_message** - The metaprompt that defines the task, behavior and tone of the LLM. 

You can change the system message to see how the LLM responds. We will cover `tools` in Lesson #4. 


In [ ]:
agent = AssistantAgent(
    name="assistant",
    model_client=client,
    tools=[],
    system_message="You are a travel agent that plans great vacations",
)

## Run the Agent 

The below function will run the agent. We use the the `on_message` method to update the Agent's state with the new message. 

In this case, we update the state with a new message from the user which is `"Plan me a great sunny vacation"`.

You can change the message content to see how the LLM responds differently. 

In [ ]:
from IPython.display import display, HTML


async def assistant_run():
    # Define the query
    user_query = "Plan me a great sunny vacation"

    # Start building HTML output
    html_output = "<div style='margin-bottom:10px'>"
    html_output += "<div style='font-weight:bold'>User:</div>"
    html_output += f"<div style='margin-left:20px'>{user_query}</div>"
    html_output += "</div>"

    try:
        # Execute the agent response
        response = await agent.on_messages(
            [TextMessage(content=user_query, source="user")],
            cancellation_token=CancellationToken(),
        )

        # Add agent response to HTML
        html_output += "<div style='margin-bottom:20px'>"
        html_output += "<div style='font-weight:bold'>Assistant:</div>"
        html_output += f"<div style='margin-left:20px; white-space:pre-wrap'>{response.chat_message.content}</div>"
        html_output += "</div>"

        # Display formatted HTML
        display(HTML(html_output))
        
    except Exception as e:
        print(f"❌ Error running agent: {e}")

# Run the function
await assistant_run()